In [1]:
import os

from dotenv import load_dotenv
from loguru import logger
from pyspark.sql import SparkSession

from arxiv_curator.arxiv_ingestion import ArxivDataIngester
from arxiv_curator.config import get_env, load_config
from arxiv_curator.utils import is_databricks

## Initialize Configuration & Spark Session

In [2]:
# Init Spark session
if not is_databricks():
    load_dotenv()
    profile = os.environ.get("PROFILE", "DEV")
    logger.info(f"Profile : {profile} (Execution outside Databricks)")

spark = SparkSession.builder.getOrCreate()
logger.info(f"Create Spark Session with the cluster: {spark.conf.get('spark.databricks.clusterUsageTags.clusterId')}")

2026-03-21 16:30:52.033 | INFO     | __main__:<module>:5 - Profile : DEV (Execution outside Databricks)
2026-03-21 16:30:52.274 | INFO     | __main__:<module>:8 - Create Spark Session with the cluster: 0321-203040-uqyjlvgl-v2n


In [3]:
# Load config
env = get_env(spark)
logger.info(f"Loading configuration (env={env})")
cfg = load_config("project_config.yml", env)
logger.info(f"Configuration loaded: catalog={cfg.catalog}schema={cfg.schema_name} processed_table={cfg.table}")

2026-03-21 16:30:55.888 | INFO     | __main__:<module>:3 - Loading configuration (env=dev)
2026-03-21 16:30:55.904 | INFO     | __main__:<module>:5 - Configuration loaded: catalog=cbcrc_catalog_devschema=rc_model_ian processed_table=arxiv_papers_corentin


In [ ]:
# Init ArxivDataIngester
logger.info("Initializing ArxivDataIngester")
ingester = ArxivDataIngester(spark, cfg)

In [ ]:
# Verify schema / Fetch arXiv papers / Create table
ingester.check_schema()
papers = ingester.fetch_papers(query="cat:cs.AI OR cat:cs.LG", max_results=50)
df_papers = ingester.create_table(papers)

In [ ]:
logger.info("Sample records:")
df_papers.select("arxiv_id", "title", "primary_category", "published").show(5, truncate=50)

In [ ]:
logger.info("Papers by primary category:")
df_papers.groupBy("primary_category").count().orderBy("count", ascending=False).show()

In [ ]:
logger.info("Most recent papers:")
df_papers.select("title", "published", "arxiv_id").orderBy("published", ascending=False).show(5, truncate=60)